# Image Detection Demo with PytorchWildlife

This tutorial guides you on how to use PyTorchWildlife to separate positive and negative animal detections. We will go through the process of setting up the environment, defining the detection model, as well as performing inference and saving the results in different ways.

## Prerequisites
Install PytorchWildlife running the following commands:
```bash
conda create -n pytorch_wildlife python=3.8 -y
conda activate pytorch_wildlife
pip install PytorchWildlife
```
Also, make sure you have a CUDA-capable GPU if you intend to run the model on a GPU. This notebook can also run on CPU.

## Importing libraries
First, we'll start by importing the necessary libraries and modules.

In [1]:
import sys, os
print("sys.executable:", sys.executable)
print("LD_LIBRARY_PATH:", os.environ.get("LD_LIBRARY_PATH", ""))

sys.executable: /home/mo/anaconda3/envs/pw20251118/bin/python
LD_LIBRARY_PATH: /usr/local/cuda-12.1/lib64:/usr/local/cuda-12.1/lib64


In [2]:
import os, sys

extra = "/usr/local/lib/python3.10/dist-packages/nvidia/nvjitlink/lib"
os.environ["LD_LIBRARY_PATH"] = extra + ":" + os.environ.get("LD_LIBRARY_PATH", "")

print("sys.executable:", sys.executable)
print("LD_LIBRARY_PATH:", os.environ["LD_LIBRARY_PATH"])

import ctypes
ctypes.CDLL("libnvJitLink.so.12")
print("Loaded libnvJitLink.so.12 OK")

sys.executable: /home/mo/anaconda3/envs/pw20251118/bin/python
LD_LIBRARY_PATH: /usr/local/lib/python3.10/dist-packages/nvidia/nvjitlink/lib:/usr/local/cuda-12.1/lib64:/usr/local/cuda-12.1/lib64
Loaded libnvJitLink.so.12 OK


In [3]:
#%pip install PytorchWildlife
import os, sys
import importlib, inspect
from pathlib import Path

# Prefer the local repo copy of PytorchWildlife (so edits in this workspace take effect).
def _find_repo_root(start_dir: str) -> str | None:
    p = Path(start_dir).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "PytorchWildlife").is_dir() and (candidate / "setup.py").is_file():
            return str(candidate)
    return None

# Try to locate the repo root from the current working directory; fall back to the known workspace path.
repo_root = _find_repo_root(os.getcwd()) or "/home/mo/CameraTraps"
local_pkg_dir = os.path.join(repo_root, "PytorchWildlife")
if not os.path.isdir(local_pkg_dir):
    raise RuntimeError(f"Local PytorchWildlife not found at {local_pkg_dir}")

# Ensure local repo path wins, and purge any already-imported installed package modules.
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
for m in list(sys.modules.keys()):
    if m == "PytorchWildlife" or m.startswith("PytorchWildlife."):
        del sys.modules[m]

import torch
from PytorchWildlife.models import detection as pw_detection
from PytorchWildlife import utils as pw_utils
import PytorchWildlife.utils.post_process as pw_post_process

# Reload to pick up any local edits
importlib.reload(pw_post_process)
importlib.reload(pw_utils)

print("PytorchWildlife loaded from:", os.path.dirname(pw_post_process.__file__))
print("post_process.detection_folder_separation:", inspect.signature(pw_post_process.detection_folder_separation))
print("pw_utils.detection_folder_separation:", inspect.signature(pw_utils.detection_folder_separation))

PytorchWildlife loaded from: /home/mo/CameraTraps/PytorchWildlife/utils
post_process.detection_folder_separation: (json_file, img_path, destination_path, confidence_threshold, output_subdir=None, copy_mode='both', preserve_relative_paths=False)
pw_utils.detection_folder_separation: (json_file, img_path, destination_path, confidence_threshold, output_subdir=None, copy_mode='both', preserve_relative_paths=False)


## Model Initialization
We will initialize the MegaDetectorV5 model for image detection. This model is designed for detecting animals in images.

In [4]:
# Ask user which device to use: GPU or CPU
choice = input("Select device [gpu/cpu] (default: gpu): " ).strip().lower()

# ---- CPU OPTION (COMMENTED OUT BY DEFAULT) ----
# if choice in ("cpu", "c"):
#     DEVICE = "cpu"
#     print("Using CPU.")
# else:

# ---- GPU OPTION (DEFAULT) ----
if choice in ("cpu", "c"):
    print("CPU option is currently disabled in this notebook. Using GPU instead.")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Please enable GPU or uncomment the CPU option.")

# Show available GPU indices
num_gpus = torch.cuda.device_count()
print(f"Number of available GPUs: {num_gpus}")
for idx in range(num_gpus):
    print(f"GPU {idx}: {torch.cuda.get_device_name(idx)}")

# Ask user which GPU index to use (default 0)
gpu_idx_input = input(f"Select GPU index [0-{num_gpus-1}] (default: 0): " ).strip()
if gpu_idx_input == "":
    gpu_idx = 0
else:
    gpu_idx = int(gpu_idx_input)
    if gpu_idx < 0 or gpu_idx >= num_gpus:
        raise ValueError(f"Invalid GPU index {gpu_idx}. Must be between 0 and {num_gpus-1}.")

DEVICE = "cuda"
torch.cuda.set_device(gpu_idx)
print(f"Using GPU: cuda:{gpu_idx}")

# Initializing the MegaDetectorV6 or V5 model for image detection
# Valid versions are MDV6-yolov9-c, MDV6-yolov9-e, MDV6-yolov10-c, MDV6-yolov10-e or MDV6-rtdetr-c
#detection_model = pw_detection.MegaDetectorV6(device=DEVICE, pretrained=True, version="MDV6-yolov10-e")

# Uncomment the following line to use MegaDetectorV5 instead of MegaDetectorV6
detection_model = pw_detection.MegaDetectorV5(device=DEVICE, pretrained=True, version="a")

Number of available GPUs: 2
GPU 0: NVIDIA GeForce RTX 4090
GPU 1: NVIDIA GeForce RTX 4090
Using GPU: cuda:1


Fusing layers... 
Model summary: 733 layers, 140054656 parameters, 0 gradients, 208.8 GFLOPs


## Variable definition
In order to process the batch detection, we will define an input directory where the images are stored, a confidence threshold and an output directory to copy the positive and negative images into distinctive folders. If you want to follow this tutorial with your own data, modify the following variables.

In [8]:
#tgt_folder_path = os.path.join(".","demo_data","imgs")
#output_path = "folder_separation"
#tgt_folder_path = "/media/mo/nvme0n1/MD_up_TT/HG"
#output_path = "/home/mo/HDD1/MD_up_TT_output/MG_MDV6-yolov10-e"
tgt_folder_path = "/mnt/sshfs_s3it/pec_filestorage/CamTrapData_General/CAM002"
#tgt_folder_path = "/home/mo/mnt/nextcloud/CamTrapData_General/CAM002"
#tgt_folder_path = "/mnt/sshfs_s3it/pec_filestorage/CamTrapData_General/CAM001/20240827"
output_path = "/media/mo/nvme0n1/20260112_MDv5a_CleanData"
threshold = 0.1 #default is 0.2

## Batch Image Detection
Next, we'll demonstrate how to process multiple images in batches. This is useful when you have a large number of images and want to process them efficiently.

In [9]:
results = detection_model.batch_image_detection(tgt_folder_path, batch_size=30) #default batch_size is 16, for 5va use 30 for cuda 1 memory reasons

100%|██████████| 9662/9662 [7:08:14<00:00,  2.66s/it]  


## Separate positive and negative detections
PytorchWildlife allows to copy the files from your original folder to a new directory containing the "Animal" and "No-animal" subdirectories. A detection is considered positive if the prediction confidence is higher than the threshold

In [10]:
# Group outputs under the site folder name (e.g., CAM001) and preserve any subfolders (e.g., 20240827/...)
tgt_folder_path = os.path.normpath(tgt_folder_path)
target_folder_name = os.path.basename(tgt_folder_path)
site_root = os.path.dirname(tgt_folder_path)
site_name = os.path.basename(site_root)

# If the selected folder looks like a date folder (YYYYMMDD), treat its parent as the site root
if target_folder_name.isdigit() and len(target_folder_name) == 8:
    site_root = os.path.dirname(tgt_folder_path)
    site_name = os.path.basename(site_root)
else:
    site_root = tgt_folder_path
    site_name = target_folder_name

output_root = os.path.join(output_path, site_name)
os.makedirs(output_root, exist_ok=True)

# Ask what to copy into the output folders
copy_mode = input("Copy which images? [both/animal/no_animal] (default: both): ").strip().lower()
if copy_mode == "":
    copy_mode = "both"
if copy_mode not in ("both", "animal", "no_animal"):
    raise ValueError("Invalid choice. Use 'both', 'animal', or 'no_animal'.")

# Save JSON with paths relative to the site root so subfolders (e.g., 20240827/...) are preserved
json_file = os.path.join(output_root, "detection_results.json")
pw_utils.save_detection_json(results, json_file,
                             categories=detection_model.CLASS_NAMES,
                             exclude_category_ids=[], # Category IDs can be found in the definition of each model.
                             exclude_file_path=site_root)

# Separate the positive and negative detections through file copying (preserve folder structure)
pw_utils.detection_folder_separation(
    json_file,
    site_root,
    output_path,
    threshold,
    output_subdir=site_name,
    copy_mode=copy_mode,
    preserve_relative_paths=True,
 )

'11566 images processed, 11143 files copied'

### Copyright (c) Microsoft Corporation. All rights reserved.
### Licensed under the MIT License.